### Structured output

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

### Pydantic

Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [5]:
import os
from langchain.chat_models import init_chat_model

os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")

model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.10'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x109541310>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x109541d10>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

### BaseModel
BaseModel is a Pydantic parent class.

It helps you create a structured data model with rules.

BaseModel = base class that gives validation + structure to your data

In [6]:
from pydantic import BaseModel, Field
class Movie(BaseModel):
    title: str = Field(description="The title of the movie")
    year: int = Field(description="The release year of the movie")
    director: str = Field( description="The director of the movie")
    rating: float = Field( description="The rating of the movie out of 10")


In [7]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.10'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x109541310>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x109541d10>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description

In [8]:
model.invoke("provide de" \
"tails about the movie Inception")

AIMessage(content='<think>\nOkay, so I need to provide details about the movie Inception. Let me start by recalling what I know. Inception is a 2010 movie directed by Christopher Nolan, right? The main cast includes Leonardo DiCaprio, Joseph Gordon-Levitt, Ellen Page, Tom Hardy, and others. The genre is sci-fi or maybe action, and it\'s known for its complex plot involving dreams. \n\nFirst, I should confirm the director and the year. I think it\'s Christopher Nolan, and the release year is definitely 2010. The title "Inception" refers to the concept of planting an idea in someone\'s subconscious. The main character, Dom Cobb, played by DiCaprio, is a thief who does this. The plot revolves around a team entering people\'s dreams to plant or extract ideas. \n\nI need to outline the main plot points. The team is led by Cobb, who has a criminal past and is trying to get his family back. The target is a business leader named Robert Fischer. The team uses technology to enter dreams, with di

In [9]:
response= model_with_structure.invoke("provide details about the movie Inception")
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

### Message output alongside Structure

parsed = clean structured data that Python can use easily

In [10]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""

    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")


# Ask model to return output in Movie structure
model_with_structure = model.with_structured_output(Movie, include_raw=True)
 #include_raw=True allows us to see the raw output from the model in addition to the structured output.
response = model_with_structure.invoke(
    "Give me details about the movie Inception"
)

response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie Inception. Let me check the tools provided. There\'s a function called Movie that requires title, year, director, and rating. I need to recall the information for Inception. The title is "Inception", the director is Christopher Nolan, it was released in 2010, and its rating is around 8.8. I should structure the tool call with these parameters. Let me make sure all required fields are included and the data types are correct. The year should be an integer, rating a number, and the director\'s name as a string. Everything looks good. Time to format the JSON accordingly.\n', 'tool_calls': [{'id': '9af42ycd3', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 186, 'prompt_tokens': 232, 'total_tokens': 418, 'completion

### Nested Structure

In [11]:
from pydantic import BaseModel, Field


class Actor(BaseModel):
    name: str
    role: str


class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(
        None,
        description="Budget in millions USD"
    )


model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke(
    "Provide details about the movie Inception"
)

response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Bane')], genres=['Action', 'Science Fiction', 'Thriller'], budget=160.0)

### TypedDict

TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.

A dictionary with fixed keys and fixed value types.

In [12]:
from typing_extensions import TypedDict, Annotated


class MovieDict(TypedDict):
    """A movie with details."""

    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]


model_with_typeddict = model.with_structured_output(MovieDict)

response = model_with_typeddict.invoke(
    "Please provide the details of the movie Avengers"
)

response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'Avengers', 'year': 2012}

In [13]:
from typing_extensions import TypedDict, Annotated, NotRequired


class Actor(TypedDict):
    name: str
    role: str


class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field( None, description="Budget in millions USD")


model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke(
    "Provide details about the movie Inception"
)

response

{'budget': 160000000,
 'cast': [{'name': 'Leonardo DiCaprio', 'role': 'Dom Cobb'},
  {'name': 'Joseph Gordon-Levitt', 'role': 'Arthur'},
  {'name': 'Ellen Page', 'role': 'Ariadne'},
  {'name': 'Tom Hardy', 'role': 'Bane'}],
 'genres': ['Action', 'Science Fiction', 'Heist'],
 'title': 'Inception',
 'year': 2010}

In [14]:
#This model.profile tells you what this LLM model can and cannot do.
model.profile

{'name': 'Qwen3 32B',
 'release_date': '2024-12-23',
 'last_updated': '2024-12-23',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 40960,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'attachment': False,
 'temperature': True}

### DataClasses

A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the @dataclass decorator

In [15]:
import os
from langchain.chat_models import init_chat_model

os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")

model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.10'}}, output_version=None, profile={'name': 'Qwen3 32B', 'release_date': '2024-12-23', 'last_updated': '2024-12-23', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 40960, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x1095e4190>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x1095e4b90>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [16]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person."""

    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")


agent = create_agent(
    model="groq:qwen/qwen3-32b",
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)


result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"
        }
    ]
})


print(result["structured_response"])
#ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')
result

name='John Doe' email='john@example.com' phone='(555) 123-4567'


{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='e70e8e50-0e0a-4867-b98d-249b5075f4c1'),
  AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user wants me to extract contact information from the given text: "John Doe, john@example.com, (555) 123-4567". Let me see. The function provided is called ContactInfo, and it requires name, email, and phone number.\n\nFirst, I need to identify each part. The name is John Doe. That\'s straightforward. The email is john@example.com. The phone number is (555) 123-4567. I should check if the phone number is in the correct format. The function doesn\'t specify a particular format, so as long as it\'s a string, it should be okay. \n\nWait, the parameters require the phone number as a string. The user included the parentheses and hyphen, so I should include that exactly as given. No need to modify it. \n\nNow, I need to struc

In [17]:
result["structured_response"] 

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [18]:
from typing_extensions import TypedDict
from langchain.agents import create_agent


class ContactInfo(TypedDict):
    """Contact information for a person."""

    name: str   # The name of the person
    email: str  # The email address of the person
    phone: str  # The phone number of the person


# If you do not have tools, keep it empty
tools = []

agent = create_agent(
    model="groq:qwen/qwen3-32b",
    tools=tools,
    response_format=ContactInfo  # Auto-selects ProviderStrategy
)


result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"
        }
    ]
})


result["structured_response"]

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

In [19]:
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str  # The name of the person
    email: str  # The email address of the person
    phone: str  # The phone number of the person

    agent = create_agent(
        model="groq:qwen/qwen3-32b",
        response_format=ContactInfo  # Auto-selects ProviderStrategy
    )

    result = agent.invoke({
        "messages": [{ "role": "user", "content": "Extract contact info from  : john Doe, john.doe@example.com, 123-456-7890" }]
    })

    result["structured_response"]  # ContactInfo(name='John Doe',